In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [5]:
def save_dataset(X, y, filename, test_ratio=0.5, val_ratio=0.5):
    n_features = X.shape[1]
    feature_names = [f"feature_{i}" for i in range(n_features)]
    X_df = pd.DataFrame(X, columns=feature_names)
    X_df["target"] = y

    # split into train and test
    train_df, test_df = train_test_split(X_df, test_size=test_ratio, random_state=42)
    val_df, test_df = train_test_split(test_df, test_size=val_ratio, random_state=42)

    print(train_df.shape, val_df.shape, test_df.shape)

    train_df.to_csv(f"../data/synthetic_cls/{filename}/train.csv", index=False)
    val_df.to_csv(f"../data/synthetic_cls/{filename}/val.csv", index=False)
    test_df.to_csv(f"../data/synthetic_cls/{filename}/test.csv", index=False)

    return train_df, val_df, test_df

### Setting 1

p = 3

\begin{equation}
    \begin{aligned}
    y = & +2 \cdot x_0 \cdot [1 + 2\tanh(x_1)] \\
    &-2 \cdot x_1 \cdot [1 + 3\sin(2x_0) + \tanh(2x_0)] \\
    &+1
    \end{aligned}
\end{equation}

In [6]:
n, p = 400, 3

np.random.seed(42)
X = np.random.uniform(-2, 2, (n, p))

beta = np.zeros(p)
beta[0] = 2
beta[1] = -2

y = X @ beta

y += X[:, 0] * beta[0] * np.tanh(X[:, 1])
y += X[:, 1] * beta[1] * 3 * np.sin(2 * X[:, 0])
y += X[:, 1] * beta[1] * np.tanh(2 * X[:, 0])

y += np.random.normal(0, 0.1, n)
y += 1

pi = 1 / (1 + np.exp(-y))
label = np.random.binomial(1, pi)

train_df, val_df, test_df = save_dataset(X, label, "setting1", test_ratio=0.5, val_ratio=0.5)

(200, 4) (100, 4) (100, 4)


### Setting 2

p = 10

\begin{equation}
    \begin{aligned}
    y = & +10 \cdot x_0 \cdot [1 + 2 \cdot \tanh(2x_1) + \sin(x_3)] \\
    &+ 20 \cdot x_1 \cdot [1 + 2 \cdot \cos(2x_0)] \\
    &-20 \cdot x_2 \cdot [1 + 2 \cdot \arctan(x_1 x_3)] \\
    & + 10 \cdot x_3 \\
    & - 10
    \end{aligned}
\end{equation}

In [ ]:
n, p = 400, 10

np.random.seed(42)
X = np.random.uniform(-2, 2, (n, p))

beta = np.zeros(p)

beta[0] = 10
beta[1] = 20
beta[2] = -20
beta[3] = 10

y = X @ beta

y += X[:, 0] * beta[0] * 2 * (np.tanh(2 * X[:, 1]) + np.sin(X[:, 3]))
y += X[:, 1] * beta[1] * 2 * np.cos(2 * X[:, 0])
y += X[:, 2] * beta[2] * 2 * np.arctan(X[:, 1] * X[:, 3])

y += np.random.normal(0, 0.1, n)
y -= 10

pi = 1 / (1 + np.exp(-y))
label = np.random.binomial(1, pi)

train_df, val_df, test_df = save_dataset(X, label, "setting2", test_ratio=0.5, val_ratio=0.5)

### Setting 3

p = 50
\begin{equation}
\begin{aligned}
y =\;& -20 \cdot x_0 \cdot [1 + \tanh(x_{1} x_{3})] \\
&+ 20 \cdot x_1 \cdot \left[1 + \frac{2}{\pi} \arctan(x_{3} - x_{4})\right] \\
&+ 30 \cdot x_3 \cdot \left[1 + \tanh(x_{1} + \sin(x_{4}))\right] \\
&-10 \cdot x_4 \cdot \left[1 + (2 \sigma(x_{0} x_{3})-1)\right]
\end{aligned}
\end{equation}

In [ ]:
n, p = 400, 50

np.random.seed(7)
X = np.random.uniform(-2, 2, (n, p))


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


beta = np.zeros(p)
beta[0] = -20
beta[1] = 20
# beta[2] = 0
beta[3] = 30
beta[4] = -10

y = X @ beta

y += X[:, 0] * beta[0] * np.tanh(X[:, 1] * X[:, 3])
y += X[:, 1] * beta[1] * 2 * np.arctan(X[:, 3] - X[:, 4]) / np.pi
y += X[:, 3] * beta[3] * np.tanh(X[:, 1] + np.sin(X[:, 4]))
y += X[:, 4] * beta[4] * (2 * sigmoid(X[:, 0] * X[:, 3]) - 1)

y += np.random.normal(0, 0.1, n)

pi = 1 / (1 + np.exp(-y))
label = np.random.binomial(1, pi)

train_df, val_df, test_df = save_dataset(X, label, "setting3", test_ratio=0.5, val_ratio=0.5)